# Feature Engineering

## Objetivo

Preparar os dados para a modelagem de previsão de demanda, criando variáveis preditoras a partir do histórico de vendas e das informações temporais, de produto, loja, eventos, SNAP e preço.
A unidade de previsão será definida por produto, loja e dia, tendo `sales` como variável-alvo.

In [3]:
import pandas as pd
import numpy as np

# ============================================================
# RECONSTRUÇÃO DA BASE DE MODELAGEM
# ============================================================

# Caminho da base original
sales_path = "../data/raw/sales_train_validation.csv"

# Identificadores necessários para a modelagem
id_cols = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

# Identifica as colunas correspondentes aos dias de venda
sales_cols = [
    col
    for col in pd.read_csv(sales_path, nrows=0).columns
    if col.startswith("d_")
]

print(f"Colunas de vendas encontradas: {len(sales_cols)}")

# Carrega somente os identificadores e as vendas
sales_wide = pd.read_csv(
    sales_path,
    usecols=id_cols + sales_cols
)

# Reduz o consumo de memória das vendas
for col in sales_cols:
    sales_wide[col] = sales_wide[col].astype("int16")

# Converte identificadores com poucos valores distintos
# para o tipo category, reduzindo o consumo de memória.
for col in [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]:
    sales_wide[col] = sales_wide[col].astype("category")

# Verificação da estrutura e do consumo de memória
print("\nDimensões:", sales_wide.shape)

print("\nMemória utilizada:")
print(
    f"{sales_wide.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

display(sales_wide.head())

Colunas de vendas encontradas: 1913

Dimensões: (30490, 1919)

Memória utilizada:
112.55 MB


,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,1,0,1,1,2,2,2,4


## Calendário de Modelagem

### Objetivo

Relacionar cada identificador de dia (`d_1`, `d_2`, ...) à sua respectiva
data e às características temporais necessárias para a modelagem.

O calendário possui apenas 1.969 registros, permitindo realizar esse
processamento de forma independente da grande base de vendas e evitando
operações desnecessárias sobre milhões de observações.

In [4]:
# ================================================================
# CALENDÁRIO DE MODELAGEM
# ================================================================

calendar_path = "../data/raw/calendar.csv"

calendar_model = pd.read_csv(
    calendar_path,
    usecols=[
        "d",
        "date",
        "wm_yr_wk",
        "wday",
        "month",
        "year"
    ]
)

# Converte a data para datetime
calendar_model["date"] = pd.to_datetime(
    calendar_model["date"]
)

# Cria a numeração sequencial do dia
calendar_model["day_num"] = np.arange(
    1,
    len(calendar_model) + 1
)

print("Dimensões:", calendar_model.shape)

print("\nPeríodo:")
print(
    calendar_model["date"].min(),
    "até",
    calendar_model["date"].max()
)

print("\nExemplo:")
display(calendar_model.head())

Dimensões: (1969, 7)

Período:
2011-01-29 00:00:00 até 2016-06-19 00:00:00

Exemplo:


,date,wm_yr_wk,wday,month,year,d,day_num
0,2011-01-29,11101,1,1,2011,d_1,1
1,2011-01-30,11101,2,1,2011,d_2,2
2,2011-01-31,11101,3,1,2011,d_3,3
3,2011-02-01,11101,4,2,2011,d_4,4
4,2011-02-02,11101,5,2,2011,d_5,5


In [5]:
# ============================================================
# FEATURES TEMPORAIS
# ============================================================


# Dia da semana em formato textual
calendar_model["weekday"] = (
    calendar_model["date"]
    .dt.day_name()
)

# Semana do ano
calendar_model["week_of_year"] = (
    calendar_model["date"]
    .dt.isocalendar()
    .week
    .astype("int8")
)

print("Features temporais criadas:")
display(
    calendar_model[
        [
            "d",
            "date",
            "day_num",
            "wday",
            "weekday",
            "month",
            "year",
            "week_of_year"
        ]
    ].head(10)
)

Features temporais criadas:


,d,date,day_num,wday,weekday,month,year,week_of_year
0,d_1,2011-01-29,1,1,Saturday,1,2011,4
1,d_2,2011-01-30,2,2,Sunday,1,2011,4
2,d_3,2011-01-31,3,3,Monday,1,2011,5
3,d_4,2011-02-01,4,4,Tuesday,2,2011,5
4,d_5,2011-02-02,5,5,Wednesday,2,2011,5
5,d_6,2011-02-03,6,6,Thursday,2,2011,5
6,d_7,2011-02-04,7,7,Friday,2,2011,5
7,d_8,2011-02-05,8,1,Saturday,2,2011,5
8,d_9,2011-02-06,9,2,Sunday,2,2011,5
9,d_10,2011-02-07,10,3,Monday,2,2011,6


## Definição da Janela de Modelagem

### Objetivo

Definir a separação temporal entre o período de treinamento e o período de
validação, respeitando a ordem cronológica das observações.

Os primeiros 28 dias serão utilizados como período de warm-up, pois as
features `lag_28` e `rolling_mean_28` dependem do histórico anterior.

Os últimos 28 dias serão reservados para validação temporal.

In [6]:
# ================================================================
# DEFINIÇÃO DA JANELA DE MODELAGEM
# ================================================================

# Total de dias disponíveis
total_days = len(sales_cols)

# Histórico necessário para as features de 28 dias
warmup_days = 28

# Horizonte de validação
validation_days = 28

# Primeiro dia elegível para modelagem
first_model_day = warmup_days + 1

# Último dia utilizado no treinamento
last_train_day = total_days - validation_days

# Período de validação
first_validation_day = last_train_day + 1
last_validation_day = total_days

print(f"Total de dias disponíveis: {total_days}")
print(f"Primeiro dia de modelagem: d_{first_model_day}")
print(f"Último dia de treino: d_{last_train_day}")
print(
    f"Período de validação: "
    f"d_{first_validation_day} até d_{last_validation_day}"
)

Total de dias disponíveis: 1913
Primeiro dia de modelagem: d_29
Último dia de treino: d_1885
Período de validação: d_1886 até d_1913


### Resultado

O histórico possui 1.913 dias.

- Warm-up: `d_1` a `d_28`
- Treinamento: `d_29` a `d_1885`
- Validação: `d_1886` a `d_1913`

### Interpretação

A divisão preserva a ordem temporal dos dados e evita que informações do
futuro sejam utilizadas durante o treinamento.

O período de warm-up garante o histórico necessário para o cálculo das
features baseadas em janelas de 28 dias.

## Construção da Base de Modelagem

### Objetivo

Transformar as vendas do formato wide para o formato longo e preparar as
variáveis defasadas necessárias à previsão.

Como a base completa possui aproximadamente 58 milhões de observações no
formato longo, o processamento será realizado por loja, evitando a criação
da estrutura completa de uma única vez na memória.

In [7]:
# ================================================================
# VERIFICAÇÃO DAS LOJAS
# ================================================================

stores = sales_wide["store_id"].unique()

print(f"Número de lojas: {len(stores)}")
print("Lojas encontradas:")
print(list(stores))

Número de lojas: 10
Lojas encontradas:
['CA_1', 'CA_2', 'CA_3', 'CA_4', 'TX_1', 'TX_2', 'TX_3', 'WI_1', 'WI_2', 'WI_3']


In [8]:
# ================================================================
# TESTE DE PROCESSAMENTO POR LOJA
# ================================================================

store_id = stores[0]

store_data = sales_wide[
    sales_wide["store_id"] == store_id
].copy()

print(f"Loja selecionada: {store_id}")
print(f"Dimensões: {store_data.shape}")

print(
    f"Memória utilizada: "
    f"{store_data.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

display(store_data.head())

Loja selecionada: CA_1
Dimensões: (3049, 1919)
Memória utilizada: 11.31 MB


,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,1,0,1,1,2,2,2,4


In [9]:
# ================================================================
# TRANSFORMAÇÃO DA LOJA PARA FORMATO LONGO
# ================================================================

sales_long = store_data.melt(
    id_vars=["item_id", "store_id"],
    value_vars=sales_cols,
    var_name="d",
    value_name="sales"
)

print("Dimensões:", sales_long.shape)

print(
    f"Memória utilizada: "
    f"{sales_long.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

display(sales_long.head())

Dimensões: (5832737, 4)
Memória utilizada: 103.23 MB


,item_id,store_id,d,sales
0,HOBBIES_1_001,CA_1,d_1,0
1,HOBBIES_1_002,CA_1,d_1,0
2,HOBBIES_1_003,CA_1,d_1,0
3,HOBBIES_1_004,CA_1,d_1,0
4,HOBBIES_1_005,CA_1,d_1,0


In [10]:
# ================================================================
# OTIMIZAÇÃO DA BASE LONGA
# ================================================================

# Converte o identificador do dia para categoria
sales_long["d"] = sales_long["d"].astype("category")

# Reduz o tipo numérico das vendas
sales_long["sales"] = sales_long["sales"].astype("int16")

print("Memória após otimização:")
print(
    f"{sales_long.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

print("\nTipos das colunas:")
print(sales_long.dtypes)

display(sales_long.head())

Memória após otimização:
39.02 MB

Tipos das colunas:
item_id     category
store_id    category
d           category
sales          int16
dtype: object


,item_id,store_id,d,sales
0,HOBBIES_1_001,CA_1,d_1,0
1,HOBBIES_1_002,CA_1,d_1,0
2,HOBBIES_1_003,CA_1,d_1,0
3,HOBBIES_1_004,CA_1,d_1,0
4,HOBBIES_1_005,CA_1,d_1,0


In [11]:
# ================================================================
# ORDENAÇÃO CRONOLÓGICA
# ================================================================

sales_long["day_num"] = (
    sales_long["d"]
    .str.extract(r"(\d+)")
    .astype("int16")
)

sales_long = sales_long.sort_values(
    ["item_id", "store_id", "day_num"]
).reset_index(drop=True)

print("Dimensões:", sales_long.shape)

display(sales_long.head(10))

Dimensões: (5832737, 5)


,item_id,store_id,d,sales,day_num
0,FOODS_1_001,CA_1,d_1,3,1
1,FOODS_1_001,CA_1,d_2,0,2
2,FOODS_1_001,CA_1,d_3,0,3
3,FOODS_1_001,CA_1,d_4,1,4
4,FOODS_1_001,CA_1,d_5,4,5
5,FOODS_1_001,CA_1,d_6,2,6
6,FOODS_1_001,CA_1,d_7,0,7
7,FOODS_1_001,CA_1,d_8,2,8
8,FOODS_1_001,CA_1,d_9,0,9
9,FOODS_1_001,CA_1,d_10,0,10


In [12]:
# ================================================================
# CRIAÇÃO DAS VARIÁVEIS DEFASADAS
# ================================================================

group_cols = ["item_id", "store_id"]

for lag in [1, 7, 14, 28]:
    sales_long[f"lag_{lag}"] = (
        sales_long
        .groupby(group_cols, observed=True)["sales"]
        .shift(lag)
    )

print("Lags criados:")
print(["lag_1", "lag_7", "lag_14", "lag_28"])

print(
    f"\nMemória utilizada: "
    f"{sales_long.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

display(
    sales_long[
        [
            "item_id",
            "store_id",
            "d",
            "sales",
            "lag_1",
            "lag_7",
            "lag_14",
            "lag_28"
        ]
    ].head(35)
)

Lags criados:
['lag_1', 'lag_7', 'lag_14', 'lag_28']

Memória utilizada: 228.15 MB


,item_id,store_id,d,sales,lag_1,lag_7,lag_14,lag_28
0,FOODS_1_001,CA_1,d_1,3,NaN,NaN,NaN,NaN
1,FOODS_1_001,CA_1,d_2,0,3.0,NaN,NaN,NaN
2,FOODS_1_001,CA_1,d_3,0,0.0,NaN,NaN,NaN
3,FOODS_1_001,CA_1,d_4,1,0.0,NaN,NaN,NaN
4,FOODS_1_001,CA_1,d_5,4,1.0,NaN,NaN,NaN
5,FOODS_1_001,CA_1,d_6,2,4.0,NaN,NaN,NaN
6,FOODS_1_001,CA_1,d_7,0,2.0,NaN,NaN,NaN
7,FOODS_1_001,CA_1,d_8,2,0.0,3.0,NaN,NaN
8,FOODS_1_001,CA_1,d_9,0,2.0,0.0,NaN,NaN
9,FOODS_1_001,CA_1,d_10,0,0.0,0.0,NaN,NaN


In [13]:
# ================================================================
# OTIMIZAÇÃO DOS LAGS
# ================================================================

lag_cols = ["lag_1", "lag_7", "lag_14", "lag_28"]

for col in lag_cols:
    sales_long[col] = sales_long[col].astype("float32")

print("Tipos dos lags:")
print(sales_long[lag_cols].dtypes)

print(
    f"\nMemória após otimização dos lags: "
    f"{sales_long.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

Tipos dos lags:
lag_1     float32
lag_7     float32
lag_14    float32
lag_28    float32
dtype: object

Memória após otimização dos lags: 139.15 MB


In [14]:
# ================================================================
# CRIAÇÃO DAS MÉDIAS MÓVEIS
# ================================================================

sales_long["rolling_mean_7"] = (
    sales_long
    .groupby(["item_id", "store_id"], observed=True)["sales"]
    .transform(
        lambda x: x.shift(1).rolling(window=7, min_periods=7).mean()
    )
    .astype("float32")
)

sales_long["rolling_mean_28"] = (
    sales_long
    .groupby(["item_id", "store_id"], observed=True)["sales"]
    .transform(
        lambda x: x.shift(1).rolling(window=28, min_periods=28).mean()
    )
    .astype("float32")
)

print("Rolling features criadas:")
print(["rolling_mean_7", "rolling_mean_28"])

print(
    f"\nMemória utilizada: "
    f"{sales_long.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

display(
    sales_long[
        [
            "item_id",
            "store_id",
            "d",
            "sales",
            "lag_7",
            "rolling_mean_7",
            "lag_28",
            "rolling_mean_28"
        ]
    ].head(35)
)

Rolling features criadas:
['rolling_mean_7', 'rolling_mean_28']

Memória utilizada: 183.65 MB


,item_id,store_id,d,sales,lag_7,rolling_mean_7,lag_28,rolling_mean_28
0,FOODS_1_001,CA_1,d_1,3,NaN,NaN,NaN,NaN
1,FOODS_1_001,CA_1,d_2,0,NaN,NaN,NaN,NaN
2,FOODS_1_001,CA_1,d_3,0,NaN,NaN,NaN,NaN
3,FOODS_1_001,CA_1,d_4,1,NaN,NaN,NaN,NaN
4,FOODS_1_001,CA_1,d_5,4,NaN,NaN,NaN,NaN
5,FOODS_1_001,CA_1,d_6,2,NaN,NaN,NaN,NaN
6,FOODS_1_001,CA_1,d_7,0,NaN,NaN,NaN,NaN
7,FOODS_1_001,CA_1,d_8,2,3.0,1.428571,NaN,NaN
8,FOODS_1_001,CA_1,d_9,0,0.0,1.285714,NaN,NaN
9,FOODS_1_001,CA_1,d_10,0,0.0,1.285714,NaN,NaN


In [15]:
# ================================================================
# LIMPEZA DA COLUNA AUXILIAR
# ================================================================

sales_long = sales_long.drop(columns="day_num")

print("Colunas atuais:")
print(sales_long.columns.tolist())

print(
    f"\nMemória após limpeza: "
    f"{sales_long.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

Colunas atuais:
['item_id', 'store_id', 'd', 'sales', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_28']

Memória após limpeza: 172.52 MB


In [16]:
# ================================================================
# VALIDAÇÃO DA BASE DE MODELAGEM
# ================================================================

print("Dimensões:", sales_long.shape)

print("\nNúmero de produtos:")
print(sales_long["item_id"].nunique())

print("\nNúmero de lojas:")
print(sales_long["store_id"].nunique())

print("\nValores ausentes:")
print(sales_long.isna().sum())

print("\nTipos das colunas:")
print(sales_long.dtypes)

Dimensões: (5832737, 10)

Número de produtos:
3049

Número de lojas:
1

Valores ausentes:
item_id                0
store_id               0
d                      0
sales                  0
lag_1               3049
lag_7              21343
lag_14             42686
lag_28             85372
rolling_mean_7     21343
rolling_mean_28    85372
dtype: int64

Tipos das colunas:
item_id            category
store_id           category
d                  category
sales                 int16
lag_1               float32
lag_7               float32
lag_14              float32
lag_28              float32
rolling_mean_7      float32
rolling_mean_28     float32
dtype: object


### Resultado

A base de modelagem da loja `CA_1` possui 5.832.737 observações,
correspondentes a 3.049 produtos ao longo de 1.913 dias.

Foram criadas as variáveis defasadas `lag_1`, `lag_7`, `lag_14` e
`lag_28`, além das médias móveis `rolling_mean_7` e `rolling_mean_28`.

Os valores ausentes concentram-se exclusivamente no início de cada
série temporal e correspondem à quantidade de dias necessários para
calcular cada variável.

A estrutura final possui 10 colunas e o consumo de memória permaneceu
controlado após a otimização dos tipos de dados.

### Interpretação

A engenharia das variáveis temporais foi realizada corretamente para
a loja analisada.

Os valores ausentes observados são esperados, pois as variáveis
defasadas e as médias móveis dependem de histórico anterior. Por
exemplo, `lag_28` e `rolling_mean_28` somente podem ser calculados
após a existência de 28 dias anteriores.

O uso de `shift(1)` nas médias móveis garante que apenas informações
disponíveis antes do dia previsto sejam utilizadas, evitando
vazamento de informação para o modelo.

A estratégia de processamento por loja também permite trabalhar com
a base sem carregar simultaneamente a estrutura completa em formato
longo na memória.


## Decisão para Modelagem

As variáveis `rolling_mean_7` e `rolling_mean_28` serão mantidas como atributos preditores.

A janela de 7 dias captura o comportamento recente e a possível sazonalidade semanal, enquanto a janela de 28 dias fornece uma referência mais estável da demanda.

## Features Temporais

### Objetivo

Criar variáveis temporais capazes de representar padrões recorrentes da demanda ao longo da semana e do ano.

### Hipótese

Espera-se que o volume de vendas apresente variações associadas ao dia da semana, à semana do ano, ao mês e ao ano, contribuindo para a previsão da demanda.

In [17]:
calendar_features = calendar_model[
    [
        "d",
        "date",
        "wday",
        "weekday",
        "month",
        "year",
        "week_of_year"
    ]
].copy()

sales_long = sales_long.merge(
    calendar_features,
    on="d",
    how="left"
)

sales_long["wday"] = sales_long["wday"].astype("int8")
sales_long["month"] = sales_long["month"].astype("int8")
sales_long["year"] = sales_long["year"].astype("int16")
sales_long["week_of_year"] = sales_long["week_of_year"].astype("int8")
sales_long["weekday"] = sales_long["weekday"].astype("category")

print("Features temporais adicionadas:")
print([
    "date",
    "wday",
    "weekday",
    "month",
    "year",
    "week_of_year"
])

print(f"\nDimensões: {sales_long.shape}")

print(
    f"Memória utilizada: "
    f"{sales_long.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

display(
    sales_long[
        [
            "item_id",
            "store_id",
            "d",
            "sales",
            "date",
            "wday",
            "weekday",
            "month",
            "year",
            "week_of_year"
        ]
    ].head(10)
)

Features temporais adicionadas:
['date', 'wday', 'weekday', 'month', 'year', 'week_of_year']

Dimensões: (5832737, 16)
Memória utilizada: 313.91 MB


,item_id,store_id,d,sales,date,wday,weekday,month,year,week_of_year
0,FOODS_1_001,CA_1,d_1,3,2011-01-29,1,Saturday,1,2011,4
1,FOODS_1_001,CA_1,d_2,0,2011-01-30,2,Sunday,1,2011,4
2,FOODS_1_001,CA_1,d_3,0,2011-01-31,3,Monday,1,2011,5
3,FOODS_1_001,CA_1,d_4,1,2011-02-01,4,Tuesday,2,2011,5
4,FOODS_1_001,CA_1,d_5,4,2011-02-02,5,Wednesday,2,2011,5
5,FOODS_1_001,CA_1,d_6,2,2011-02-03,6,Thursday,2,2011,5
6,FOODS_1_001,CA_1,d_7,0,2011-02-04,7,Friday,2,2011,5
7,FOODS_1_001,CA_1,d_8,2,2011-02-05,1,Saturday,2,2011,5
8,FOODS_1_001,CA_1,d_9,0,2011-02-06,2,Sunday,2,2011,5
9,FOODS_1_001,CA_1,d_10,0,2011-02-07,3,Monday,2,2011,6


In [18]:
# ============================================================
# VALIDAÇÃO E OTIMIZAÇÃO DAS FEATURES TEMPORAIS
# ============================================================

print("Tipos após o merge:")
print(sales_long.dtypes)

print(
    f"\nMemória antes da otimização: "
    f"{sales_long.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

# Otimiza os tipos das variáveis temporais
sales_long["wday"] = sales_long["wday"].astype("int8")
sales_long["month"] = sales_long["month"].astype("int8")
sales_long["year"] = sales_long["year"].astype("int16")
sales_long["week_of_year"] = sales_long["week_of_year"].astype("int8")
sales_long["weekday"] = sales_long["weekday"].astype("category")

print(
    f"\nMemória após a otimização: "
    f"{sales_long.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

print("\nTipos após a otimização:")
print(sales_long.dtypes)

Tipos após o merge:
item_id                  category
store_id                 category
d                             str
sales                       int16
lag_1                     float32
lag_7                     float32
lag_14                    float32
lag_28                    float32
rolling_mean_7            float32
rolling_mean_28           float32
date               datetime64[us]
wday                         int8
weekday                  category
month                        int8
year                        int16
week_of_year                 int8
dtype: object

Memória antes da otimização: 313.91 MB

Memória após a otimização: 313.91 MB

Tipos após a otimização:
item_id                  category
store_id                 category
d                             str
sales                       int16
lag_1                     float32
lag_7                     float32
lag_14                    float32
lag_28                    float32
rolling_mean_7            float32
rolling_mean

In [19]:
# Verifica os tipos após o merge

print("Tipos após o merge:")
print(sales_long.dtypes)

print(
    f"\nMemória antes da otimização: "
    f"{sales_long.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

Tipos após o merge:
item_id                  category
store_id                 category
d                             str
sales                       int16
lag_1                     float32
lag_7                     float32
lag_14                    float32
lag_28                    float32
rolling_mean_7            float32
rolling_mean_28           float32
date               datetime64[us]
wday                         int8
weekday                  category
month                        int8
year                        int16
week_of_year                 int8
dtype: object

Memória antes da otimização: 313.91 MB


### Resultado

As informações temporais do calendário foram incorporadas à base de modelagem, associando cada observação diária às respectivas características de data, semana, dia da semana, mês e ano.

### Interpretação

As variáveis temporais permitem representar padrões recorrentes da demanda relacionados ao calendário. A utilização do calendário previamente preparado também evita recalcular essas informações durante o processamento da base.

### Decisão para Modelagem

As variáveis temporais serão mantidas como atributos candidatos à modelagem. A variável `date` será utilizada como referência temporal, enquanto as demais características poderão ser utilizadas como variáveis preditoras.

## SNAP e Eventos

### Objetivo

Incorporar informações sobre eventos e a participação no programa SNAP para representar possíveis variações externas na demanda.

### Hipótese

Espera-se que eventos e períodos com ativação do SNAP estejam associados a alterações no comportamento das vendas, podendo contribuir para a previsão da demanda.


In [20]:
# ============================================================
# SNAP E EVENTOS
# ============================================================

calendar_events = pd.read_csv(
    calendar_path,
    usecols=[
        "d",
        "event_name_1",
        "event_type_1",
        "event_name_2",
        "event_type_2",
        "snap_CA",
        "snap_TX",
        "snap_WI"
    ]
)

sales_long = sales_long.merge(
    calendar_events,
    on="d",
    how="left"
)

print("Informações de eventos e SNAP adicionadas.")

print(f"\nDimensões: {sales_long.shape}")

display(
    sales_long[
        [
            "item_id",
            "store_id",
            "d",
            "sales",
            "event_name_1",
            "event_type_1",
            "event_name_2",
            "event_type_2",
            "snap_CA",
            "snap_TX",
            "snap_WI"
        ]
    ].head(10)
)

Informações de eventos e SNAP adicionadas.

Dimensões: (5832737, 23)


,item_id,store_id,d,sales,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,FOODS_1_001,CA_1,d_1,3,NaN,NaN,NaN,NaN,0,0,0
1,FOODS_1_001,CA_1,d_2,0,NaN,NaN,NaN,NaN,0,0,0
2,FOODS_1_001,CA_1,d_3,0,NaN,NaN,NaN,NaN,0,0,0
3,FOODS_1_001,CA_1,d_4,1,NaN,NaN,NaN,NaN,1,1,0
4,FOODS_1_001,CA_1,d_5,4,NaN,NaN,NaN,NaN,1,0,1
5,FOODS_1_001,CA_1,d_6,2,NaN,NaN,NaN,NaN,1,1,1
6,FOODS_1_001,CA_1,d_7,0,NaN,NaN,NaN,NaN,1,0,0
7,FOODS_1_001,CA_1,d_8,2,NaN,NaN,NaN,NaN,1,1,1
8,FOODS_1_001,CA_1,d_9,0,SuperBowl,Sporting,NaN,NaN,1,1,1
9,FOODS_1_001,CA_1,d_10,0,NaN,NaN,NaN,NaN,1,1,0


In [21]:
# ============================================================
# VALIDAÇÃO E OTIMIZAÇÃO DE SNAP E EVENTOS
# ============================================================

print("Tipos das novas colunas:")
print(
    sales_long[
        [
            "event_name_1",
            "event_type_1",
            "event_name_2",
            "event_type_2",
            "snap_CA",
            "snap_TX",
            "snap_WI"
        ]
    ].dtypes
)

print("\nValores ausentes:")
print(
    sales_long[
        [
            "event_name_1",
            "event_type_1",
            "event_name_2",
            "event_type_2",
            "snap_CA",
            "snap_TX",
            "snap_WI"
        ]
    ].isna().sum()
)

Tipos das novas colunas:
event_name_1      str
event_type_1      str
event_name_2      str
event_type_2      str
snap_CA         int64
snap_TX         int64
snap_WI         int64
dtype: object

Valores ausentes:
event_name_1    5363191
event_type_1    5363191
event_name_2    5820541
event_type_2    5820541
snap_CA               0
snap_TX               0
snap_WI               0
dtype: int64


In [22]:
# ============================================================
# OTIMIZAÇÃO DAS VARIÁVEIS DE SNAP E EVENTOS
# ============================================================

# Variáveis categóricas
event_cols = [
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

for col in event_cols:
    sales_long[col] = sales_long[col].astype("category")

# Variáveis SNAP
snap_cols = [
    "snap_CA",
    "snap_TX",
    "snap_WI"
]

for col in snap_cols:
    sales_long[col] = sales_long[col].astype("int8")

print("Tipos após otimização:")
print(
    sales_long[
        event_cols + snap_cols
    ].dtypes
)

print(
    f"\nMemória utilizada: "
    f"{sales_long.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

Tipos após otimização:
event_name_1    category
event_type_1    category
event_name_2    category
event_type_2    category
snap_CA             int8
snap_TX             int8
snap_WI             int8
dtype: object

Memória utilizada: 352.85 MB


### Resultado

As informações de eventos e participação no programa SNAP foram incorporadas à base de modelagem sem alteração do número de observações.

As variáveis de eventos foram convertidas para o tipo categórico e as variáveis SNAP foram otimizadas para `int8`, mantendo os indicadores de participação como variáveis numéricas.

### Interpretação

A presença de eventos permite representar possíveis alterações externas no comportamento da demanda, enquanto as variáveis SNAP representam a participação do estabelecimento no programa ao longo do tempo.

Os valores ausentes nas variáveis de eventos são esperados, pois a maioria dos dias não possui eventos registrados. As variáveis SNAP não apresentam valores ausentes.

### Decisão para Modelagem

As variáveis de eventos e SNAP serão mantidas como candidatas a variáveis preditoras.

As informações de eventos poderão contribuir para capturar variações pontuais da demanda, enquanto os indicadores SNAP poderão representar efeitos associados ao programa de assistência alimentar.

## SNAP e Eventos

### Objetivo

Incorporar informações sobre eventos e a participação no programa SNAP à base de modelagem, permitindo representar possíveis variações externas no comportamento da demanda.

### Hipótese

Espera-se que eventos e períodos com ativação do SNAP estejam associados a alterações no comportamento das vendas, podendo contribuir para a previsão da demanda.

In [23]:
# Cria a variável SNAP correspondente ao estado da loja

state_id = sales_long["store_id"].iloc[0][:2]
snap_col = f"snap_{state_id}"

sales_long["snap"] = sales_long[snap_col].astype("int8")

print(f"Loja processada: {sales_long['store_id'].iloc[0]}")
print(f"Estado identificado: {state_id}")
print(f"Coluna SNAP utilizada: {snap_col}")

display(
    sales_long[
        ["store_id", snap_col, "snap"]
    ].head(20)
)

Loja processada: CA_1
Estado identificado: CA
Coluna SNAP utilizada: snap_CA


,store_id,snap_CA,snap
0,CA_1,0,0
1,CA_1,0,0
2,CA_1,0,0
3,CA_1,1,1
4,CA_1,1,1
5,CA_1,1,1
6,CA_1,1,1
7,CA_1,1,1
8,CA_1,1,1
9,CA_1,1,1


In [24]:
# ============================================================
# VALIDAÇÃO DA VARIÁVEL SNAP
# ============================================================

print("Tipo da variável SNAP:")
print(sales_long["snap"].dtype)

print("\nDistribuição dos valores SNAP:")
print(sales_long["snap"].value_counts().sort_index())

print("\nValores ausentes:")
print(sales_long["snap"].isna().sum())

print("\nSNAP por loja:")
print(
    sales_long.groupby("store_id")["snap"]
    .value_counts()
    .sort_index()
)


Tipo da variável SNAP:
int8

Distribuição dos valores SNAP:
snap
0    3911867
1    1920870
Name: count, dtype: int64

Valores ausentes:
0

SNAP por loja:
store_id  snap
CA_1      0       3911867
          1       1920870
Name: count, dtype: int64


In [25]:
# Validação rápida da variável SNAP

display(
    sales_long[
        ["store_id", "snap"]
    ].head(20)
)

,store_id,snap
0,CA_1,0
1,CA_1,0
2,CA_1,0
3,CA_1,1
4,CA_1,1
5,CA_1,1
6,CA_1,1
7,CA_1,1
8,CA_1,1
9,CA_1,1


### Resultado

A variável SNAP foi consolidada a partir do indicador correspondente ao estado da loja em processamento, mantendo o número de observações da base.

A variável foi armazenada como `int8`, sem valores ausentes, representando corretamente a participação da loja no programa SNAP ao longo do período analisado.

### Interpretação

A variável SNAP apresenta comportamento binário, distinguindo os períodos com e sem participação no programa.

A ausência de valores nulos indica que o indicador está disponível para todo o período da loja em processamento. Como a base é processada uma loja por vez, o indicador consolidado utiliza exclusivamente o SNAP correspondente ao estado da loja.

### Decisão para Modelagem

A variável `snap` será mantida como candidata a variável preditora.

Sua utilização permitirá avaliar se os períodos de participação no programa SNAP contribuem para explicar variações no comportamento das vendas e melhorar a previsão da demanda.

## Eventos

### Objetivo

Incorporar informações sobre eventos ao conjunto de dados, permitindo representar possíveis variações pontuais ou externas no comportamento da demanda.

### Hipótese

Espera-se que a ocorrência de eventos esteja associada a alterações no comportamento das vendas em determinados períodos, podendo contribuir para a previsão da demanda.

In [26]:
# ================================================================
# EVENTOS
# ================================================================

# Carrega somente as variáveis de eventos necessárias
calendar_events = pd.read_csv(
    calendar_path,
    usecols=[
        "d",
        "event_name_1",
        "event_type_1",
        "event_name_2",
        "event_type_2"
    ]
)

# Integra os eventos à base da loja atual
sales_long = sales_long.merge(
    calendar_events,
    on="d",
    how="left"
)

print("Informações de eventos adicionadas.")
print(f"Dimensões: {sales_long.shape}")

Informações de eventos adicionadas.
Dimensões: (5832737, 28)


In [27]:
# ================================================================
# EVENTOS
# ================================================================

event_cols = [
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

# Remove as colunas caso esta etapa já tenha sido executada anteriormente
sales_long = sales_long.drop(
    columns=[col for col in event_cols if col in sales_long.columns]
)

# Carrega os eventos do calendário
calendar_events = pd.read_csv(
    calendar_path,
    usecols=[
        "d",
        "event_name_1",
        "event_type_1",
        "event_name_2",
        "event_type_2"
    ]
)

# Integra os eventos à loja atualmente processada
sales_long = sales_long.merge(
    calendar_events,
    on="d",
    how="left"
)

print("Eventos adicionados.")
print(f"Dimensões: {sales_long.shape}")

print("\nColunas de eventos presentes:")
print([col for col in event_cols if col in sales_long.columns])

Eventos adicionados.
Dimensões: (5832737, 32)

Colunas de eventos presentes:
['event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']


In [28]:
# ================================================================
# VALIDAÇÃO DOS EVENTOS
# ================================================================

print("Tipos das variáveis de eventos:")
print(
    sales_long[
        [
            "event_name_1",
            "event_type_1",
            "event_name_2",
            "event_type_2"
        ]
    ].dtypes
)

print("\nValores ausentes:")
print(
    sales_long[
        [
            "event_name_1",
            "event_type_1",
            "event_name_2",
            "event_type_2"
        ]
    ].isna().sum()
)

display(
    sales_long[
        [
            "d",
            "store_id",
            "event_name_1",
            "event_type_1",
            "event_name_2",
            "event_type_2"
        ]
    ].head(20)
)

Tipos das variáveis de eventos:
event_name_1    str
event_type_1    str
event_name_2    str
event_type_2    str
dtype: object

Valores ausentes:
event_name_1    5363191
event_type_1    5363191
event_name_2    5820541
event_type_2    5820541
dtype: int64


,d,store_id,event_name_1,event_type_1,event_name_2,event_type_2
0,d_1,CA_1,NaN,NaN,NaN,NaN
1,d_2,CA_1,NaN,NaN,NaN,NaN
2,d_3,CA_1,NaN,NaN,NaN,NaN
3,d_4,CA_1,NaN,NaN,NaN,NaN
4,d_5,CA_1,NaN,NaN,NaN,NaN
5,d_6,CA_1,NaN,NaN,NaN,NaN
6,d_7,CA_1,NaN,NaN,NaN,NaN
7,d_8,CA_1,NaN,NaN,NaN,NaN
8,d_9,CA_1,SuperBowl,Sporting,NaN,NaN
9,d_10,CA_1,NaN,NaN,NaN,NaN


In [29]:
# ================================================================
# OTIMIZAÇÃO DAS VARIÁVEIS DE EVENTOS
# ================================================================

event_cols = [
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

for col in event_cols:
    sales_long[col] = sales_long[col].astype("category")

print("Tipos após otimização:")
print(sales_long[event_cols].dtypes)

Tipos após otimização:
event_name_1    category
event_type_1    category
event_name_2    category
event_type_2    category
dtype: object


In [30]:
# ================================================================
# CONFERÊNCIA DE SNAP E EVENTOS
# ================================================================

display(
    sales_long[
        [
            "d",
            "store_id",
            "sales",
            "snap",
            "event_name_1",
            "event_type_1",
            "event_name_2",
            "event_type_2"
        ]
    ].head(20)
)

,d,store_id,sales,snap,event_name_1,event_type_1,event_name_2,event_type_2
0,d_1,CA_1,3,0,NaN,NaN,NaN,NaN
1,d_2,CA_1,0,0,NaN,NaN,NaN,NaN
2,d_3,CA_1,0,0,NaN,NaN,NaN,NaN
3,d_4,CA_1,1,1,NaN,NaN,NaN,NaN
4,d_5,CA_1,4,1,NaN,NaN,NaN,NaN
5,d_6,CA_1,2,1,NaN,NaN,NaN,NaN
6,d_7,CA_1,0,1,NaN,NaN,NaN,NaN
7,d_8,CA_1,2,1,NaN,NaN,NaN,NaN
8,d_9,CA_1,0,1,SuperBowl,Sporting,NaN,NaN
9,d_10,CA_1,0,1,NaN,NaN,NaN,NaN


### Resultado

As informações de eventos foram incorporadas à base da loja em processamento, mantendo o número de observações.

As variáveis de identificação dos eventos foram convertidas para o tipo categórico, reduzindo o consumo de memória e preservando sua informação para utilização na modelagem.

### Interpretação

Os valores ausentes nas variáveis de eventos são esperados, pois representam os dias em que não há evento registrado no calendário.

A presença de eventos permite representar possíveis variações pontuais na demanda que não seriam capturadas apenas pelas características temporais.

### Decisão para Modelagem

As variáveis de eventos serão mantidas como candidatas a variáveis preditoras.

A contribuição efetiva dessas variáveis será avaliada durante a etapa de modelagem e validação do modelo.

## Lags da Demanda

### Objetivo

Criar variáveis que representem o histórico recente de vendas de cada produto, utilizando valores observados em dias anteriores.

Os lags permitem que o modelo utilize o comportamento passado da demanda como informação para prever os períodos seguintes.

Serão considerados os atrasos de 1, 7, 14 e 28 dias.

In [31]:
# ============================================================
# LAGS DA DEMANDA
# ============================================================

# Garante a ordenação temporal dos registros
sales_long["day_num"] = (
    sales_long["d"]
    .str.extract(r"(\d+)")
    .astype("int16")
)

sales_long = sales_long.sort_values(
    ["item_id", "store_id", "day_num"]
).reset_index(drop=True)

# Cria os lags da demanda
lag_days = [1, 7, 14, 28]

for lag in lag_days:
    sales_long[f"lag_{lag}"] = (
        sales_long
        .groupby(
            ["item_id", "store_id"],
            observed=True
        )["sales"]
        .shift(lag)
        .astype("float32")
    )

print("Lags criados:")
print([f"lag_{lag}" for lag in lag_days])

print("\nDimensões:", sales_long.shape)

display(
    sales_long[
        [
            "d",
            "item_id",
            "store_id",
            "sales",
            "lag_1",
            "lag_7",
            "lag_14",
            "lag_28"
        ]
    ].head(40)
)

Lags criados:
['lag_1', 'lag_7', 'lag_14', 'lag_28']

Dimensões: (5832737, 33)


,d,item_id,store_id,sales,lag_1,lag_7,lag_14,lag_28
0,d_1,FOODS_1_001,CA_1,3,NaN,NaN,NaN,NaN
1,d_2,FOODS_1_001,CA_1,0,3.0,NaN,NaN,NaN
2,d_3,FOODS_1_001,CA_1,0,0.0,NaN,NaN,NaN
3,d_4,FOODS_1_001,CA_1,1,0.0,NaN,NaN,NaN
4,d_5,FOODS_1_001,CA_1,4,1.0,NaN,NaN,NaN
5,d_6,FOODS_1_001,CA_1,2,4.0,NaN,NaN,NaN
6,d_7,FOODS_1_001,CA_1,0,2.0,NaN,NaN,NaN
7,d_8,FOODS_1_001,CA_1,2,0.0,3.0,NaN,NaN
8,d_9,FOODS_1_001,CA_1,0,2.0,0.0,NaN,NaN
9,d_10,FOODS_1_001,CA_1,0,0.0,0.0,NaN,NaN


In [32]:
# ============================================================
# VALIDAÇÃO DOS LAGS
# ============================================================

lag_cols = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28"
]

print("Tipos dos lags:")
print(sales_long[lag_cols].dtypes)

print("\nValores ausentes:")
print(sales_long[lag_cols].isna().sum())

print("\nMemória utilizada pelos lags:")
print(
    f"{sales_long[lag_cols].memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

Tipos dos lags:
lag_1     float32
lag_7     float32
lag_14    float32
lag_28    float32
dtype: object

Valores ausentes:
lag_1      3049
lag_7     21343
lag_14    42686
lag_28    85372
dtype: int64

Memória utilizada pelos lags:
89.00 MB


### Resultado

Foram criadas quatro variáveis defasadas de vendas: `lag_1`, `lag_7`, `lag_14` e `lag_28`, utilizando os valores históricos da própria série de cada combinação de produto e loja.

As variáveis foram armazenadas como `float32`, mantendo o consumo de memória controlado.

### Interpretação

A quantidade de valores ausentes aumenta conforme o tamanho do lag, o que é esperado, pois não existem observações históricas suficientes no início de cada série para calcular essas variáveis.

O `lag_1` representa a venda do dia anterior, enquanto `lag_7`, `lag_14` e `lag_28` representam, respectivamente, as vendas de 7, 14 e 28 dias anteriores. Essas informações permitem ao modelo capturar dependências temporais e padrões recentes da demanda.

### Decisão para Modelagem

As variáveis `lag_1`, `lag_7`, `lag_14` e `lag_28` serão mantidas como candidatas a variáveis preditoras.

As observações sem histórico suficiente permanecerão ausentes nesta etapa, sendo tratadas posteriormente de acordo com a estratégia definida para a construção da base de treinamento e validação.

## Médias Móveis (Rolling)

### Objetivo

Incorporar informações sobre o comportamento recente das vendas à base de modelagem, utilizando médias móveis de diferentes janelas para representar o nível médio da demanda ao longo do tempo.

### Hipótese

Espera-se que o comportamento médio das vendas nos períodos anteriores apresente relação com a demanda futura, permitindo ao modelo capturar tendências e padrões recentes que não são representados apenas pelas características temporais e pelos valores individuais dos lags.

In [33]:
# ============================================================
# FEATURES DE MÉDIA MÓVEL (ROLLING)
# ============================================================

# Cria uma ordenação numérica temporária para os dias
sales_long["_day_num"] = (
    sales_long["d"]
    .str.extract(r"(\d+)")
    .astype("int32")
)

# Ordena cronologicamente cada série de produto e loja
sales_long = sales_long.sort_values(
    ["item_id", "store_id", "_day_num"]
).reset_index(drop=True)

# Remove rolling anteriores, caso já existam
rolling_cols = [
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28"
]

sales_long = sales_long.drop(
    columns=[col for col in rolling_cols if col in sales_long.columns]
)

# Média dos 7 dias anteriores
sales_long["rolling_mean_7"] = (
    sales_long
    .groupby(["item_id", "store_id"])["sales"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=7,
            min_periods=7
        ).mean()
    )
    .astype("float32")
)

# Média dos 14 dias anteriores
sales_long["rolling_mean_14"] = (
    sales_long
    .groupby(["item_id", "store_id"])["sales"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=14,
            min_periods=14
        ).mean()
    )
    .astype("float32")
)

# Média dos 28 dias anteriores
sales_long["rolling_mean_28"] = (
    sales_long
    .groupby(["item_id", "store_id"])["sales"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=28,
            min_periods=28
        ).mean()
    )
    .astype("float32")
)

print("Features rolling criadas:")
print([
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28"
])

print("\nDimensões:", sales_long.shape)

display(
    sales_long[
        [
            "d",
            "item_id",
            "store_id",
            "sales",
            "rolling_mean_7",
            "rolling_mean_14",
            "rolling_mean_28"
        ]
    ].head(35)
)

Features rolling criadas:
['rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28']

Dimensões: (5832737, 35)


,d,item_id,store_id,sales,rolling_mean_7,rolling_mean_14,rolling_mean_28
0,d_1,FOODS_1_001,CA_1,3,NaN,NaN,NaN
1,d_2,FOODS_1_001,CA_1,0,NaN,NaN,NaN
2,d_3,FOODS_1_001,CA_1,0,NaN,NaN,NaN
3,d_4,FOODS_1_001,CA_1,1,NaN,NaN,NaN
4,d_5,FOODS_1_001,CA_1,4,NaN,NaN,NaN
5,d_6,FOODS_1_001,CA_1,2,NaN,NaN,NaN
6,d_7,FOODS_1_001,CA_1,0,NaN,NaN,NaN
7,d_8,FOODS_1_001,CA_1,2,1.428571,NaN,NaN
8,d_9,FOODS_1_001,CA_1,0,1.285714,NaN,NaN
9,d_10,FOODS_1_001,CA_1,0,1.285714,NaN,NaN


In [34]:
# ============================================================
# VALIDAÇÃO DAS FEATURES ROLLING
# ============================================================

rolling_cols = [
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28"
]

print("Tipos das features rolling:")
print(sales_long[rolling_cols].dtypes)

print("\nValores ausentes:")
print(sales_long[rolling_cols].isna().sum())

print("\nMemória utilizada pelas features rolling:")
print(
    f"{sales_long[rolling_cols].memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

Tipos das features rolling:
rolling_mean_7     float32
rolling_mean_14    float32
rolling_mean_28    float32
dtype: object

Valores ausentes:
rolling_mean_7     21343
rolling_mean_14    42686
rolling_mean_28    85372
dtype: int64

Memória utilizada pelas features rolling:
66.75 MB


### Resultado

As features de média móvel foram criadas com janelas de 7, 14 e 28 dias, utilizando apenas os valores anteriores ao dia de previsão.

As três variáveis foram armazenadas como `float32`, mantendo o consumo de memória controlado. Os valores ausentes observados no início das séries são esperados, pois correspondem ao período necessário para formar cada janela histórica.

### Interpretação

As médias móveis representam diferentes horizontes do comportamento recente das vendas.

A janela de 7 dias captura variações de curto prazo, enquanto as janelas de 14 e 28 dias representam padrões de médio prazo. A utilização de `shift(1)` garante que a venda do próprio dia não seja utilizada no cálculo da variável, evitando vazamento de informação.

### Decisão para Modelagem

As features `rolling_mean_7`, `rolling_mean_14` e `rolling_mean_28` serão mantidas como candidatas a variáveis preditoras.

A contribuição de cada janela será avaliada durante a etapa de treinamento e validação do modelo.

## Validação das Features de Histórico

### Objetivo

Avaliar a consistência das variáveis de histórico criadas para representar o comportamento passado das vendas, verificando tipos, valores ausentes e consumo de memória.

### Hipótese

As variáveis de lag e média móvel devem apresentar valores ausentes apenas no início de cada série, devido à necessidade de histórico prévio para o cálculo. Espera-se também que os tipos numéricos e o consumo de memória permaneçam adequados para a construção da base final.

A utilização de períodos anteriores ao dia de previsão deve evitar vazamento de informação para a modelagem.

In [35]:
# ============================================================
# VALIDAÇÃO CONJUNTA DAS FEATURES DE HISTÓRICO
# ============================================================

lag_cols = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28"
]

rolling_cols = [
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28"
]

history_cols = lag_cols + rolling_cols

print("Tipos das features de histórico:")
print(sales_long[history_cols].dtypes)

print("\nValores ausentes:")
print(sales_long[history_cols].isna().sum())

print("\nMemória utilizada pelas features de histórico:")
print(
    f"{sales_long[history_cols].memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

print("\nResumo das features:")
display(
    sales_long[
        ["d", "item_id", "store_id", "sales"] + history_cols
    ].head(35)
)

Tipos das features de histórico:
lag_1              float32
lag_7              float32
lag_14             float32
lag_28             float32
rolling_mean_7     float32
rolling_mean_14    float32
rolling_mean_28    float32
dtype: object

Valores ausentes:
lag_1               3049
lag_7              21343
lag_14             42686
lag_28             85372
rolling_mean_7     21343
rolling_mean_14    42686
rolling_mean_28    85372
dtype: int64

Memória utilizada pelas features de histórico:
155.75 MB

Resumo das features:


,d,item_id,store_id,sales,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_14,rolling_mean_28
0,d_1,FOODS_1_001,CA_1,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,d_2,FOODS_1_001,CA_1,0,3.0,NaN,NaN,NaN,NaN,NaN,NaN
2,d_3,FOODS_1_001,CA_1,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,d_4,FOODS_1_001,CA_1,1,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,d_5,FOODS_1_001,CA_1,4,1.0,NaN,NaN,NaN,NaN,NaN,NaN
5,d_6,FOODS_1_001,CA_1,2,4.0,NaN,NaN,NaN,NaN,NaN,NaN
6,d_7,FOODS_1_001,CA_1,0,2.0,NaN,NaN,NaN,NaN,NaN,NaN
7,d_8,FOODS_1_001,CA_1,2,0.0,3.0,NaN,NaN,1.428571,NaN,NaN
8,d_9,FOODS_1_001,CA_1,0,2.0,0.0,NaN,NaN,1.285714,NaN,NaN
9,d_10,FOODS_1_001,CA_1,0,0.0,0.0,NaN,NaN,1.285714,NaN,NaN


### Features de histórico

Foram criadas defasagens de 1, 7, 14 e 28 dias e médias móveis de 7, 14 e 28 dias para representar diferentes padrões de dependência temporal das vendas.

Os valores ausentes concentram-se no início das séries, devido à necessidade de histórico para o cálculo das respectivas janelas. Esse comportamento é esperado.

As features foram armazenadas como `float32` para manter o consumo de memória controlado e serão disponibilizadas para a etapa de modelagem.

## Base Final

### Objetivo

Consolidar as variáveis necessárias para a modelagem da demanda em uma única base, garantindo consistência estrutural, temporal e de tipos antes da separação entre treino e validação.

In [36]:
print("Colunas atuais do sales_long:")
print(sales_long.columns.tolist())

print("\nDimensões:")
print(sales_long.shape)

Colunas atuais do sales_long:
['item_id', 'store_id', 'd', 'sales', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'date', 'wday', 'weekday', 'month', 'year', 'week_of_year', 'event_name_1_x', 'event_type_1_x', 'event_name_2_x', 'event_type_2_x', 'snap_CA', 'snap_TX', 'snap_WI', 'snap', 'event_name_1_y', 'event_type_1_y', 'event_name_2_y', 'event_type_2_y', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'day_num', '_day_num', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28']

Dimensões:
(5832737, 35)


In [37]:
# ============================================================
# VALIDAÇÃO ESTRUTURAL DA BASE FINAL
# ============================================================

print("Dimensões da base final:")
print(sales_long.shape)

print("\nNúmero de linhas duplicadas:")
print(sales_long.duplicated().sum())

print("\nTipos das principais variáveis:")
print(sales_long.dtypes)

print("\nValores ausentes:")
print(
    sales_long.isna().sum()
    .sort_values(ascending=False)
)

print("\nIdentificadores únicos:")
print(f"Itens: {sales_long['item_id'].nunique()}")
print(f"Lojas: {sales_long['store_id'].nunique()}")
print(f"Dias: {sales_long['d'].nunique()}")

print("\nPeríodo:")
print(f"Primeiro dia: {sales_long['date'].min()}")
print(f"Último dia: {sales_long['date'].max()}")

Dimensões da base final:
(5832737, 35)

Número de linhas duplicadas:
0

Tipos das principais variáveis:
item_id                  category
store_id                 category
d                             str
sales                       int16
lag_1                     float32
lag_7                     float32
lag_14                    float32
lag_28                    float32
date               datetime64[us]
wday                         int8
weekday                  category
month                        int8
year                        int16
week_of_year                 int8
event_name_1_x           category
event_type_1_x           category
event_name_2_x           category
event_type_2_x           category
snap_CA                      int8
snap_TX                      int8
snap_WI                      int8
snap                         int8
event_name_1_y                str
event_type_1_y                str
event_name_2_y                str
event_type_2_y                str
event_name_1

In [38]:
# ============================================================
# CONFERÊNCIA FINAL DAS COLUNAS E VALORES AUSENTES
# ============================================================

print("Colunas relacionadas a eventos:")
print([
    col for col in sales_long.columns
    if "event" in col
])

print("\nColunas duplicadas por sufixo:")
print([
    col for col in sales_long.columns
    if col.endswith("_x") or col.endswith("_y")
])

print("\nValores ausentes:")
missing = sales_long.isna().sum()
print(missing[missing > 0].sort_values(ascending=False))

Colunas relacionadas a eventos:
['event_name_1_x', 'event_type_1_x', 'event_name_2_x', 'event_type_2_x', 'event_name_1_y', 'event_type_1_y', 'event_name_2_y', 'event_type_2_y', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']

Colunas duplicadas por sufixo:
['event_name_1_x', 'event_type_1_x', 'event_name_2_x', 'event_type_2_x', 'event_name_1_y', 'event_type_1_y', 'event_name_2_y', 'event_type_2_y']

Valores ausentes:
event_name_2_y     5820541
event_type_2_x     5820541
event_name_2_x     5820541
event_name_2       5820541
event_type_2       5820541
event_type_2_y     5820541
event_type_1_x     5363191
event_type_1       5363191
event_type_1_y     5363191
event_name_1_y     5363191
event_name_1_x     5363191
event_name_1       5363191
lag_28               85372
rolling_mean_28      85372
lag_14               42686
rolling_mean_14      42686
lag_7                21343
rolling_mean_7       21343
lag_1                 3049
dtype: int64


In [39]:
# ============================================================
# LIMPEZA DAS COLUNAS DUPLICADAS DE EVENTOS
# ============================================================

event_duplicate_cols = [
    "event_name_1_x",
    "event_type_1_x",
    "event_name_2_x",
    "event_type_2_x",
    "event_name_1_y",
    "event_type_1_y",
    "event_name_2_y",
    "event_type_2_y"
]

sales_long = sales_long.drop(columns=event_duplicate_cols)

print("Colunas de eventos mantidas:")
print([
    col for col in sales_long.columns
    if col.startswith("event_")
])

print("\nDimensões da base:")
print(sales_long.shape)

Colunas de eventos mantidas:
['event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']

Dimensões da base:
(5832737, 27)


In [40]:
print(sales_long.shape)
print([col for col in sales_long.columns if "event" in col])

(5832737, 27)
['event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']


In [41]:
# ============================================================
# VALIDAÇÃO PÓS-LIMPEZA DA BASE FINAL
# ============================================================

print("Dimensões da base final:")
print(sales_long.shape)

print("\nColunas de eventos mantidas:")
print([
    col
    for col in sales_long.columns
    if col.startswith("event_")
])

print("\nColunas duplicadas por sufixo:")
print([
    col
    for col in sales_long.columns
    if col.endswith("_x") or col.endswith("_y")
])

Dimensões da base final:
(5832737, 27)

Colunas de eventos mantidas:
['event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']

Colunas duplicadas por sufixo:
[]


In [42]:
print("Valores ausentes nas features históricas:")

history_cols = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28"
]

print(sales_long[history_cols].isna().sum())

Valores ausentes nas features históricas:
lag_1               3049
lag_7              21343
lag_14             42686
lag_28             85372
rolling_mean_7     21343
rolling_mean_14    42686
rolling_mean_28    85372
dtype: int64


In [43]:
print("Valores ausentes nas features históricas:")

history_cols = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28"
]

print(sales_long[history_cols].isna().sum())

Valores ausentes nas features históricas:
lag_1               3049
lag_7              21343
lag_14             42686
lag_28             85372
rolling_mean_7     21343
rolling_mean_14    42686
rolling_mean_28    85372
dtype: int64


### Fechamento da Engenharia de Features

A base foi consolidada após a criação e validação das variáveis históricas, de calendário e de eventos.

- Linhas: 5.832.737
- Variáveis: 27
- Linhas duplicadas: 0
- Variáveis históricas: `lag_1`, `lag_7`, `lag_14`, `lag_28`, `rolling_mean_7`, `rolling_mean_14` e `rolling_mean_28`
- Variáveis de calendário e eventos mantidas para utilização na modelagem.

A base está estruturada e validada para seguir para a etapa de preparação dos dados.

In [44]:
sales_long.to_parquet(
    "../data/processed/sales_features.parquet",
    index=False
)

In [45]:
print("Base de features salva com sucesso.")
print(f"Dimensões: {sales_long.shape}")
print("Arquivo: ../data/processed/sales_features.parquet")

Base de features salva com sucesso.
Dimensões: (5832737, 27)
Arquivo: ../data/processed/sales_features.parquet
